In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import darkprop as dp
from darkprop import units
from darkprop import constants

rn = dp.RandomNumber(1);
inv_cdf_r_file = "../solardm/data/inverseCDFIntd2Ngammadtdr.csv"
inv_cdf_T_file = "../solardm/data/inverseCDFIntdSFactordE-mdm1.00075MeV.csv"
density_integral_file = "../solardm/data/normalizedDensityIntegral.csv"

# Scattering

In [ ]:
number_of_samples = 100000

sigma = 1e-32 * units.cm2
mchi = 1.0 * units.keV
dm = dp.SolarDM(sigma, mchi, t=0)

r = 200.0 * units.km
init_r = [0.0, 0.0, r]
init_p = [0.0, 0.0, 1.0 * units.MeV]
target = dp.Target("proton", 1, 1, constants.mp)

Ts = np.empty(number_of_samples)

for i in range(number_of_samples):
    dm.setR(init_r)
    dm.setP(init_p)
    dm.scatter(target, rn)
    Ts[i] = dm.T

fig, ax = plt.subplots()
ax.hist(Ts / units.MeV, bins=100, density=True);
ax.set_xlabel("T [MeV]")
ax.set_ylabel("f(T) [MeV$^{-1}$]")

# Free Path

In [ ]:
number_of_samples = 100000

sigma = 1e-32 * units.cm2
mchi = 1.0 * units.MeV
dm = dp.SolarDM(sigma, mchi)

sun = dp.Sun()
sun.init(density_integral_file)

y = 0.0
init_r = [0.0, 0.0, y * constants.rSun]
init_p = [1.0, 0.0, 0.0]

freeps = np.zeros(number_of_samples)

dm.setP(init_p)

for i in range(number_of_samples):
    dm.setR(init_r)
    sun.propagate(dm, rn)

    freeps[i] = (dm.r - init_r).norm()

fig, ax = plt.subplots()
ax.hist(freeps / units.km, bins=100, density=True)
ax.set_yscale("log")
ax.set_xlabel("$\lambda$ [km]")
ax.set_ylabel("$f(\lambda)$ [km$^{-1}$]");

# Injection

In [ ]:
number_of_samples = 100000

sigma = 1e-32 * units.cm2
mchi = 1.0 * units.MeV
dm = dp.SolarDM(sigma, mchi)

injector = dp.SourceInjector(inv_cdf_r_file, inv_cdf_T_file)


rs = []
Ts = []
speeds = []
for i in range(number_of_samples):
    injector.next(dm)

    rs.append(np.array(dm.r.to_vec()))
    Ts.append(dm.T)

rs = np.array(rs)
Ts = np.array(Ts)

## Show initial positions

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(projection='3d')
x = rs[:, 0] / units.km
y = rs[:, 1] / units.km
z = rs[:, 2] / units.km

ax.scatter(x, y, z, s=0.05)
ax.set_xlabel('x [km]')
ax.set_ylabel('y [km]')
ax.set_zlabel('z [km]')
;

## Show initial kinetic energy

In [ ]:
fig, ax = plt.subplots()
ax.hist(Ts / units.MeV, bins=100, density=True)
ax.set_yscale("log")
ax.set_xlabel("$T$ [MeV]")
ax.set_ylabel("$f(T)$ [MeV$^{-1}$]");

# Trajectory

In [ ]:
number_of_tracks = 10
sigma = 1e-34 * units.cm2
mchi = 1.0 * units.MeV
Tcut = 1 * units.keV

dm = dp.SolarDM(sigma, mchi)
sun = dp.Sun(density_integral_file)

init_r = [0.0, 0.0, 0.0]
init_p = [0.0, 0.0, 5.0 * units.MeV]

tracks = []

for i in range(number_of_tracks):
    dm.in_medium = True
    dm.setP(init_p)
    dm.setR(init_r)

    tracks.append(dp.simulate_track(dm, sun, Tcut, rn))

fig = plt.figure()
ax = fig.add_subplot(projection="3d")
for track in tracks:
    n = len(track)
    x = np.empty(n)
    y = np.empty(n)
    z = np.empty(n)
    for i in range(n):
        r = track[i].r.to_vec()
        x[i] = r[0]
        y[i] = r[1]
        z[i] = r[2]

    ax.scatter(x, y, z, s=0.1)
    ax.plot(x, y, z, ls="-", lw=0.5)

ax.set_xlabel("x [km]")
ax.set_ylabel("y [km]")
ax.set_zlabel("z [km]")
;